In [1]:
import NN_model_helper
import numpy as np, pandas as pd, optuna, torch
from sklearn.model_selection import StratifiedKFold
from NN_model import ImprovedNN 
from pathlib import Path
import json, torch
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from kneed import KneeLocator
from NN_model_helper import (evaluate_fold, plot_training_progress, find_optimal_clusters)

import sys
from pathlib import Path

/opt/anaconda3/envs/Surabie_S_clean_v2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# classifier/ → MELTING_POINT_2026/
PROJECT_ROOT = Path.cwd().parent        # directory above a path: .../MELTING_POINT_2026

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

processed_dir = PROJECT_ROOT / "data_curation" / "processed_data"

data_path = processed_dir / "final_dataset.parquet"
df = pd.read_parquet(data_path)

print("Loaded:", data_path)
print("Shape:", df.shape)

Loaded: /Users/sdl5_mp/Documents/GitHub/melting_point_2026/data_curation/processed_data/final_dataset.parquet
Shape: (17220, 122)


In [3]:
df.head()

,SMILES,MP,Type,Ro5,RDKit_PEOE_VSA3,RDKit_NumAliphaticRings X RDKit_SlogP_VSA8,RDKit_NHOHCount,RDKit_SMR_VSA10 X RDKit_VSA_EState6,MACCS_105 X RDKit_NHOHCount,MACCS_105 X RDKit_fr_Ar_COO,...,MACCS_161,RDKit_NHOHCount X RDKit_SlogP_VSA8,RDKit_BertzCT X RDKit_RingCount,RDKit_fr_ArN,RDKit_SMR_VSA5,MACCS_126 X RDKit_NumHDonors,RDKit_MaxAbsPartialCharge,MACCS_155 X RDKit_fr_Ar_OH,RDKit_Chi3n X RDKit_NumRotatableBonds,RDKit_SlogP_VSA2
0,ON=Cc1cscc1,122.0,Train,1,0.0,0.0,1,32.985231,0,0,...,1,0.0,168.564167,0,0.000000,0,0.410848,0,0.749989,11.421854
1,O=C1CC[C@]2(C(=C1)CC[C@@H]1[C@@H]2[C@H](O)C[C@...,205.5,Train,1,0.0,0.0,1,0.000000,1,0,...,0,0.0,3553.423052,0,77.105125,0,0.392773,0,8.850244,34.482001
2,[O-][n+]1ccccc1,64.0,Train,1,0.0,0.0,0,0.000000,0,0,...,1,0.0,138.007504,0,0.000000,0,0.618694,0,0.000000,0.000000
3,OC1CCC2(C(C1)CC=C1C2CCC2(C1CCC2C(CCC(C(C)C)C)C...,146.0,Train,1,0.0,0.0,1,0.000000,1,0,...,0,0.0,2479.213445,0,111.854606,0,0.393120,0,55.138771,11.210494
4,CC(=O)c1ccc(cc1)Br,51.0,Train,1,0.0,0.0,0,158.762615,0,0,...,0,0.0,237.939433,0,6.923737,0,0.294512,0,1.250748,5.783245


In [ ]:
df.describe()

,MP,Ro5,RDKit_PEOE_VSA3,RDKit_NumAliphaticRings X RDKit_SlogP_VSA8,RDKit_NHOHCount,RDKit_SMR_VSA10 X RDKit_VSA_EState6,MACCS_105 X RDKit_NHOHCount,MACCS_105 X RDKit_fr_Ar_COO,RDKit_FpDensityMorgan1 X RDKit_NumRotatableBonds,RDKit_NHOHCount X RDKit_fr_Ar_COO,...,MACCS_161,RDKit_NHOHCount X RDKit_SlogP_VSA8,RDKit_BertzCT X RDKit_RingCount,RDKit_fr_ArN,RDKit_SMR_VSA5,MACCS_126 X RDKit_NumHDonors,RDKit_MaxAbsPartialCharge,MACCS_155 X RDKit_fr_Ar_OH,RDKit_Chi3n X RDKit_NumRotatableBonds,RDKit_SlogP_VSA2
count,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,...,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000
mean,126.040571,0.978688,3.025448,0.976507,1.097909,138.151883,0.290012,0.006620,2.838261,0.085017,...,0.593670,2.691309,1215.041434,0.084959,19.166147,0.240244,0.394746,0.035075,10.562985,18.358919
std,70.914541,0.144428,4.629141,5.091234,1.305635,224.123470,0.824084,0.088625,2.475850,0.561846,...,0.491162,10.297792,1639.353590,0.313165,27.956990,0.768413,0.113005,0.298360,22.831053,16.826626
min,0.000000,0.000000,0.000000,0.000000,0.000000,-942.806447,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,-0.061902,0.000000,0.000000,0.000000,0.000000,-0.061902
25%,69.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.181818,0.000000,...,0.000000,0.000000,270.794610,0.000000,0.000000,0.000000,0.325709,0.000000,1.258586,7.109798
50%,120.000000,1.000000,0.000000,0.000000,1.000000,69.508109,0.000000,0.000000,2.400000,0.000000,...,1.000000,0.000000,628.851074,0.000000,6.923737,0.000000,0.417985,0.000000,4.346087,14.703796
75%,175.000000,1.000000,4.794537,0.000000,2.000000,174.859973,0.000000,0.000000,4.090909,0.000000,...,1.000000,0.000000,1606.391404,0.000000,25.683286,0.000000,0.481173,0.000000,11.397890,23.662430
max,492.500000,1.000000,65.856226,133.522836,25.000000,6374.741017,11.000000,4.000000,20.460000,36.000000,...,1.000000,266.155326,65644.728169,4.000000,335.074616,25.000000,1.000000,25.000000,694.930816,224.669012


In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
from sklearn.preprocessing import StandardScaler

# 2) Filter to TRAIN only
df_test = df[df["Type"].astype(str).str.lower() == "test"].copy()

# 3) Drop non-feature columns
NON_FEATURES = ["SMILES", "MP", "Type", "Ro5"]
feature_cols = [c for c in df_test.columns if c not in NON_FEATURES]

# (prevents issues if any non-numeric columns sneak in)
feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df_test[c])]

X_test = df_test[feature_cols].to_numpy(dtype=np.float32)

BASE = Path.cwd()
scaler_path = BASE / "artifacts" / "df_final_scaler.pkl"
scaler = joblib.load(scaler_path)

X_test_scaled = scaler.transform(X_test)

# 7) Rebuild DataFrame with SAME column order
df_test_scaled = df_test.copy()
df_test_scaled[feature_cols] = X_test_scaled

# 8) Save as Parquet
out_path = BASE / "artifacts" / "df_test_scaled.parquet"
df_test_scaled.to_parquet(out_path, index=False)

print("Test rows:", df_test_scaled.shape[0])
print("Num features:", len(feature_cols))
print("Saved scaled TEST dataset to:", out_path)

Test rows: 5166
Num features: 118
Saved scaled TEST dataset to: /Users/sdl5_mp/Documents/GitHub/melting_point_2026/Ro5/artifacts/df_test_scaled.parquet


In [6]:

df_test_scaled.head()

,SMILES,MP,Type,Ro5,RDKit_PEOE_VSA3,RDKit_NumAliphaticRings X RDKit_SlogP_VSA8,RDKit_NHOHCount,RDKit_SMR_VSA10 X RDKit_VSA_EState6,MACCS_105 X RDKit_NHOHCount,MACCS_105 X RDKit_fr_Ar_COO,...,MACCS_161,RDKit_NHOHCount X RDKit_SlogP_VSA8,RDKit_BertzCT X RDKit_RingCount,RDKit_fr_ArN,RDKit_SMR_VSA5,MACCS_126 X RDKit_NumHDonors,RDKit_MaxAbsPartialCharge,MACCS_155 X RDKit_fr_Ar_OH,RDKit_Chi3n X RDKit_NumRotatableBonds,RDKit_SlogP_VSA2
12054,[O-][N+](=O)c1c(C)c(C(=O)C)c(c(c1C(C)(C)C)[N+]...,135.5,Test,1,-0.651134,-0.195586,-0.839721,-0.618301,-0.350082,-0.077211,...,0.821926,-0.258763,-0.385976,-0.268728,0.998128,-0.316404,-0.871431,-0.145223,-0.020079,-0.165679
12055,CN(NC(=O)CCC(=O)O)C,154.5,Test,1,0.432729,-0.195586,0.694148,-0.618301,-0.350082,-0.077211,...,0.821926,-0.258763,-0.775069,-0.268728,-0.226208,-0.316404,0.764344,-0.145223,-0.315950,1.029142
12056,CCCCc1ccc(cc1)NC(=O)Oc1ccc(cc1)OC,143.0,Test,1,0.386341,-0.195586,-0.072787,0.145154,-0.350082,-0.077211,...,0.821926,-0.258763,-0.027957,-0.268728,0.252697,1.006958,0.900505,-0.145223,0.410717,-0.307419
12057,OC(=O)COCCN1C(=O)c2c(C1=O)cccc2,128.0,Test,1,0.386341,-0.195586,-0.072787,-0.103564,0.862879,-0.077211,...,0.821926,-0.258763,-0.173134,-0.268728,-0.687065,1.006958,0.749574,-0.145223,0.097925,1.698546
12058,CCCCCCCCCCCCCCC,10.0,Test,1,-0.651134,-0.195586,-0.839721,-0.618301,-0.350082,-0.077211,...,-1.216654,-0.258763,-0.775069,-0.268728,2.805458,-0.316404,-2.878269,-0.145223,1.258066,-1.078546


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
from sklearn.preprocessing import StandardScaler

# 2) Filter to TRAIN only
df_train = df[df["Type"].astype(str).str.lower() == "train"].copy()

# 3) Drop non-feature columns
NON_FEATURES = ["SMILES", "MP", "Type", "Ro5"]
feature_cols = [c for c in df_train.columns if c not in NON_FEATURES]

# (optional) keep only numeric feature columns
# (prevents issues if any non-numeric columns sneak in)
feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df_train[c])]

X_train = df_train[feature_cols].to_numpy(dtype=np.float32)

# 4) Fit scaler on TRAIN features only
scaler = StandardScaler()
scaler.fit(X_train)

# 5) Save scaler to ./artifacts (relative to the notebook)
BASE = Path.cwd()
artifacts_dir = BASE / "artifacts"
artifacts_dir.mkdir(parents=True, exist_ok=True)

scaler_path = artifacts_dir / "df_final_scaler.pkl"
joblib.dump(scaler, scaler_path)

print("Train rows:", len(df_train))
print("Num features:", len(feature_cols))
print("Scaler saved to:", scaler_path)